# Strategic Sourcing Optimization
This notebook demonstrates a procurement optimization model using supplier and demand data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pulp import LpMinimize, LpProblem, LpVariable, lpSum

## Load Datasets

In [ ]:
suppliers = pd.read_csv('suppliers.csv')
costs = pd.read_csv('costs.csv')
demand = pd.read_csv('demand.csv')

suppliers.head(), costs.head(), demand.head()

## Supplier Scoring

In [ ]:
suppliers['CostScore'] = 100 - (suppliers['LeadTime'] / suppliers['LeadTime'].max())*100
suppliers['TotalScore'] = 0.5*suppliers['ReliabilityScore'] + 0.3*suppliers['SustainabilityScore'] + 0.2*suppliers['CostScore']
suppliers[['SupplierName','ReliabilityScore','SustainabilityScore','CostScore','TotalScore']]

## Optimization Model
We want to minimize total procurement cost while meeting demand and respecting supplier constraints.

In [ ]:
model = LpProblem(name='sourcing', sense=LpMinimize)

# Decision variables: how much to buy from each supplier for each material
x = {(row.SupplierID, row.Material): LpVariable(name=f"x_{row.SupplierID}_{row.Material}", 
                                                lowBound=0, upBound=row.MaxCapacity)
     for idx, row in costs.iterrows()}

# Objective: minimize cost
model += lpSum(row.UnitCost * x[(row.SupplierID, row.Material)] for idx, row in costs.iterrows())

# Constraints: meet demand for each material (total across months)
for mat in demand['Material'].unique():
    total_demand = demand.loc[demand['Material']==mat, 'DemandQty'].sum()
    model += lpSum(x[(row.SupplierID, row.Material)] 
                   for idx, row in costs[costs['Material']==mat].iterrows()) >= total_demand, f"demand_{mat}"

# Solve
model.solve()

# Results
allocation = []
for (sup, mat), var in x.items():
    if var.value() > 0:
        allocation.append([sup, mat, var.value()])

alloc_df = pd.DataFrame(allocation, columns=['Supplier','Material','Qty'])
alloc_df

## Visualization

In [ ]:
alloc_df.groupby('Supplier')['Qty'].sum().plot(kind='bar', title='Total Allocation by Supplier')
plt.ylabel('Quantity')
plt.show()